In [36]:
import requests 
import pandas as pd 
import time

SEASON = "20252026"
BASE   = "https://api-web.nhle.com/v1"
HEADERS = {"User-Agent": "Mozilla/5.0"}

In [37]:
def get_all_teams():
    url = f"{BASE}/standings/now"
    r = requests.get(url, headers=HEADERS)
    r.raise_for_status()
    standings = r.json()["standings"]
    teams = [t["teamAbbrev"]["default"] for t in standings]
    print(f"Found {len(teams)} teams: {teams}")
    return teams

In [38]:

# In a new cell:
def get_season_schedule():
    """
    The NHL schedule endpoint returns every game for the season.
    We iterate month by month to avoid hitting response size limits.
    """
    all_games = []
    # Regular season runs Oct 2025 – Apr 2026
    next_date = "2025-10-01"
    end_date = "2026-04-30"
 
    while next_date and next_date <= end_date:
        url = f"{BASE}/schedule/{next_date}"
        r = requests.get(url, headers=HEADERS)
        if r.status_code != 200:
            print(f"  Skipping {month_start}: HTTP {r.status_code}")
            continue
 
        data = r.json()
        weeks = data.get("gameWeek")
        if not isinstance(weeks, list):
            weeks = []
        for week in weeks:
            if not isinstance(week, dict):
                continue
            games = week.get("games")
            if not isinstance(games, list):
                games = []
            for game in games:
                if not isinstance(game, dict):
                    continue
                # Only regular season (gameType 2) 
                if game.get("gameType") not in (2,):
                    continue
                # Safely access keys with defaults
                game_date = game.get("gameDate", "Unknown")
                home_score = game.get("homeTeam", {}).get("score")
                away_score = game.get("awayTeam", {}).get("score")
                venue = game.get("venue", {}).get("default", "")
                all_games.append({
                    "game_id":       game.get("id"),
                    "date":          game_date,
                    "game_type":     "regular" if game.get("gameType") == 2 else "playoff",
                    "home_team":     game.get("homeTeam", {}).get("abbrev"),
                    "away_team":     game.get("awayTeam", {}).get("abbrev"),
                    "home_score":    home_score,
                    "away_score":    away_score,
                    "game_state":    game.get("gameState"),  # OFF = final
                    "venue":         venue,
                })

        next_date = data.get("nextStartDate")
        print(f"  {next_date}: {len(all_games)} games so far")
        time.sleep(0.3)
 
    df = pd.DataFrame(all_games)
    # Keep only completed games
    df = df[df["game_state"] == "OFF"].reset_index(drop=True)
    print(f"\nTotal completed games: {len(df)}")
    return df

# In a new cell:


In [39]:
df = get_season_schedule()
df.head()

  2025-10-08: 3 games so far
  2025-10-15: 56 games so far
  2025-10-22: 107 games so far
  2025-10-29: 165 games so far
  2025-11-05: 212 games so far
  2025-11-12: 264 games so far
  2025-11-19: 314 games so far
  2025-11-26: 360 games so far
  2025-12-03: 420 games so far
  2025-12-10: 475 games so far
  2025-12-17: 528 games so far
  2025-12-24: 587 games so far
  2025-12-31: 621 games so far
  2026-01-07: 676 games so far
  2026-01-14: 734 games so far
  2026-01-21: 787 games so far
  2026-01-28: 838 games so far
  2026-02-04: 891 games so far
  2026-02-11: 908 games so far
  2026-02-18: 908 games so far
  2026-02-25: 908 games so far
  2026-03-04: 968 games so far
  2026-03-11: 1024 games so far
  2026-03-18: 1076 games so far
  2026-03-25: 1134 games so far
  2026-04-01: 1187 games so far
  2026-04-08: 1243 games so far
  2026-04-15: 1300 games so far
  2026-04-22: 1312 games so far
  2026-04-29: 1312 games so far
  None: 1312 games so far

Total completed games: 1312


,game_id,date,game_type,home_team,away_team,home_score,away_score,game_state,venue
0,2025020001,Unknown,regular,FLA,CHI,3,2,OFF,Amerant Bank Arena
1,2025020002,Unknown,regular,NYR,PIT,0,3,OFF,Madison Square Garden
2,2025020003,Unknown,regular,LAK,COL,1,4,OFF,Crypto.com Arena
3,2025020004,Unknown,regular,TOR,MTL,5,2,OFF,Scotiabank Arena
4,2025020005,Unknown,regular,WSH,BOS,1,3,OFF,Capital One Arena


In [ ]:
def team_level_game_stats(game):
    import requests
    import pandas as pd

    game_id = game["game_id"]

    url = f"https://api.nhle.com/stats/rest/en/team/summary?cayenneExp=gameId={game_id}"
    response = requests.get(url)

    if response.status_code != 200:
        print(f"Failed {game_id}: {response.status_code}")
        return pd.DataFrame()

    data = response.json().get("data", [])

    rows = []

    for team in data:
        team_id = team.get("teamId")

        rows.append({
            "game_id": game_id,
            "game_date": game.get("game_date", None),

            "team_id": team_id,
            "team_name": team.get("teamFullName"),

            "goals_for": team.get("goalsFor"),
            "goals_against": team.get("goalsAgainst"),

            "shots_for": team.get("shotsForPerGame"),
            "shots_against": team.get("shotsAgainstPerGame"),

            "faceoff_win_pct": team.get("faceoffWinPct"),
            "power_play_pct": team.get("powerPlayPct"),
            "penalty_kill_pct": team.get("penaltyKillPct"),

            "wins": team.get("wins"),
            "losses": team.get("losses"),
            "points": team.get("points"),
            "point_pct": team.get("pointPct"),
            "team_shutouts": team.get("teamShutouts")
        })

    return pd.DataFrame(rows)

In [42]:
import requests
import pandas as pd

def get_all_games(start_date="2025-10-01"):
    games = []
    current_date = start_date
    seen_dates = set()

    while current_date not in seen_dates:
        seen_dates.add(current_date)

        url = f"https://api-web.nhle.com/v1/schedule/{current_date}"
        response = requests.get(url)

        if response.status_code != 200:
            print(f"Failed at {current_date}: {response.status_code}")
            break

        data = response.json()

        for day in data.get("gameWeek", []):
            game_date = day.get("date")

            for game in day.get("games", []):
                games.append({
                    "game_id": game.get("id"),
                    "game_date": game_date,
                    "home_team_id": game.get("homeTeam", {}).get("id"),
                    "away_team_id": game.get("awayTeam", {}).get("id"),
                    "home_team": game.get("homeTeam", {}).get("abbrev"),
                    "away_team": game.get("awayTeam", {}).get("abbrev")
                })

        next_date = data.get("nextStartDate")

        if not next_date:
            break

        current_date = next_date

    return pd.DataFrame(games)

In [43]:
import time
import pandas as pd

all_game_stats = []

for i, game in df.iterrows():
    game_df = team_level_game_stats(game)

    if not game_df.empty:
        all_game_stats.append(game_df)

    if i % 50 == 0:
        print(f"Processed {i} games")

    time.sleep(0.05)

df_team_stats = pd.concat(all_game_stats, ignore_index=True)

df_team_stats = df_team_stats.sort_values(["game_id", "team_id"])

df_team_stats.to_csv("nhl_team_gamelog_2025_26.csv", index=False)

df_team_stats.head()

Processed 0 games
Processed 50 games
Processed 100 games
Processed 150 games
Processed 200 games
Processed 250 games
Processed 300 games
Processed 350 games
Processed 400 games
Processed 450 games
Processed 500 games
Processed 550 games
Processed 600 games
Processed 650 games
Processed 700 games
Processed 750 games
Processed 800 games
Processed 850 games
Processed 900 games
Processed 950 games
Processed 1000 games
Processed 1050 games
Processed 1100 games
Processed 1150 games
Processed 1200 games
Processed 1250 games
Processed 1300 games


C:\Users\mnpol\AppData\Local\Temp\ipykernel_24992\3464440258.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_team_stats = pd.concat(all_game_stats, ignore_index=True)


,game_id,game_date,team_id,team_name,goals_for,goals_against,shots_for,shots_against,faceoff_win_pct,power_play_pct,penalty_kill_pct,wins,losses,points,point_pct,team_shutouts
1,2025020001,None,13,Florida Panthers,3,2,37.0,19.0,0.585714,0.500000,1.00,1,0,2,1.0,0
0,2025020001,None,16,Chicago Blackhawks,2,3,19.0,37.0,0.414285,0.000000,0.50,0,1,0,0.0,0
3,2025020002,None,3,New York Rangers,0,3,25.0,31.0,0.475000,0.000000,1.00,0,1,0,0.0,0
2,2025020002,None,5,Pittsburgh Penguins,3,0,31.0,25.0,0.525000,0.000000,1.00,1,0,2,1.0,1
4,2025020003,None,21,Colorado Avalanche,4,1,23.0,25.0,0.450000,0.166666,0.75,1,0,2,1.0,0


In [1]:
import requests
import pandas as pd



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\mnpol\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\mnpol\AppData\Roaming\Python\Python311\site-packages\traitlets\config\application.py", line 1043, in launch_instance
    app.start()
  File "C:\Users\mnpol\AppData\Roaming\Python\Python311\site-packages\ipykernel\kernelapp.py", line 725, in start
    self.io_lo

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\mnpol\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\mnpol\AppData\Roaming\Python\Python311\site-packages\traitlets\config\application.py", line 1043, in launch_instance
    app.start()
  File "C:\Users\mnpol\AppData\Roaming\Python\Python311\site-packages\ipykernel\kernelapp.py", line 725, in start
    self.io_lo

AttributeError: _ARRAY_API not found

In [7]:
df= pd.read_csv("nhl_team_gamelog_2025_26.csv")
df = df.drop(columns=['game_date'])
df.head()

,game_id,team_id,team_name,goals_for,goals_against,shots_for,shots_against,faceoff_win_pct,power_play_pct,penalty_kill_pct,wins,losses,points,point_pct,team_shutouts
0,2025020001,13,Florida Panthers,3,2,37.0,19.0,0.585714,0.500000,1.00,1,0,2,1.0,0
1,2025020001,16,Chicago Blackhawks,2,3,19.0,37.0,0.414285,0.000000,0.50,0,1,0,0.0,0
2,2025020002,3,New York Rangers,0,3,25.0,31.0,0.475000,0.000000,1.00,0,1,0,0.0,0
3,2025020002,5,Pittsburgh Penguins,3,0,31.0,25.0,0.525000,0.000000,1.00,1,0,2,1.0,1
4,2025020003,21,Colorado Avalanche,4,1,23.0,25.0,0.450000,0.166666,0.75,1,0,2,1.0,0


In [ ]:
def player_level_game_stats(game):
    game_id = str(int(df["game_id"].iloc[0]))

    url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/boxscore"
    response = requests.get(url)

    if response.status_code != 200:
        print(f"Failed to fetch boxscore for game {game_id}: HTTP {response.status_code}")
        return pd.DataFrame()
    try:
        data = response.json()
    except:
        print(f"Failed to parse JSON for game {game_id}")
        return pd.DataFrame()
    player_stats = data.get("playerByGameStats", {})
    
    if not isinstance(player_stats, dict):
        print(f"Unexpected playerStats format for game {game_id}")
        return pd.DataFrame()
    rows = []

    for team_key in ["homeTeam", "awayTeam"]:
        team_data = player_stats.get(team_key, {})
        team_meta = data.get(team_key, {})
        team_id = team_meta.get("id")
        
        for position in ["forwards", "defensemen", "goalies"]:
            players = team_data.get(position, [])

            for player in players:
                rows.append({
                    "game_id": game_id,
                    "team_id": team_id,
                    "home_away": "home" if team_key == "homeTeam" else "away",
                    "player_position": player.get("position"),  # "skater" or "goalie"
                    "player_id": player.get("playerId"),
                    "player_name": player.get("name",{}).get("default"),

                    "goals": player.get("goals", 0),
                    "assists": player.get("assists", 0),
                    "points": player.get("points", 0),
                    "shots": player.get("sog", 0),
                    "time_on_ice": player.get("toi", 0),
                    "hits": player.get("hits", 0),
                    "blocked_shots": player.get("blockedShots", 0),
                    "giveaways": player.get("giveaways", 0),
                    "takeaways": player.get("takeaways", 0),
                    "pims": player.get("pim", 0),
                })
    
    return pd.DataFrame(rows)
    

In [44]:
player_game_stats = player_level_game_stats(df.iloc[0])
player_game_stats.head()

Game 2025020001 player stats keys: ['awayTeam', 'homeTeam']
dict_keys(['forwards', 'defense', 'goalies'])
dict_keys(['forwards', 'defense', 'goalies'])


,game_id,team_id,home_away,player_position,player_id,player_name,goals,assists,points,shots,time_on_ice,hits,blocked_shots,giveaways,takeaways,pims
0,2025020001,13,home,C,8477935,S. Bennett,0,0,0,4,19:15,1,0,0,1,0
1,2025020001,13,home,L,8478421,A. Greer,1,0,1,1,07:19,5,1,0,0,5
2,2025020001,13,home,R,8482713,M. Samoskevich,0,2,2,0,11:18,3,0,0,0,0
3,2025020001,13,home,L,8479981,J. Gadjovich,0,1,1,1,07:18,4,1,0,1,0
4,2025020001,13,home,C,8477933,S. Reinhart,0,0,0,3,18:14,1,2,2,0,0
